# wandb-log-step — faded example 2: Log metrics continuing from a non-zero examples_seen

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-log-step`. Running the beacon reports progress on the `Logging: wandb.log step` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.log step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-log-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-log-step"
DD_SUBTOPIC = "Logging: wandb.log step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

When resuming training or logging a second phase, the `examples_seen` counter does not reset to zero — it continues from where the previous phase ended. The `step=` argument to `wandb.log` must reflect the total examples seen since the start of the entire training run, not just the current phase.

## Faded exercise 2

Implement `log_second_phase(losses, batch_size, start_examples)`. The counter starts at `start_examples` (already accumulated from a previous phase) and continues from there. For each loss, increment and log as usual. Return the final `examples_seen`.

Complete the blank that initializes the counter from `start_examples` and logs each step.

**Fill in:** Set examples_seen = start_examples, then for each loss increment by batch_size and call wandb.log({'loss': loss}, step=examples_seen).

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def log_second_phase(losses, batch_size, start_examples):
    raise NotImplementedError()  # TODO: Set examples_seen = start_examples, then for each loss increment by batch_size and call wandb.log({'loss': loss}, step=examples_seen).
    return examples_seen

wandb.log.reset_mock()
final = log_second_phase([0.5, 0.4, 0.3], batch_size=32, start_examples=640)
print('final examples_seen:', final)  # 640 + 3*32 = 736
print('first log step:', wandb.log.call_args_list[0].kwargs['step'])  # 672


import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def log_second_phase(losses, batch_size, start_examples):
    examples_seen = start_examples
    for loss in losses:
        examples_seen += batch_size
        wandb.log({'loss': loss}, step=examples_seen)
    return examples_seen

def _test():
    wandb.log.reset_mock()
    losses = [0.5, 0.4, 0.3]
    bs = 32
    start = 640
    final = log_second_phase(losses, bs, start)
    assert final == start + len(losses) * bs
    calls = wandb.log.call_args_list
    assert calls[0].kwargs['step'] == start + bs
    assert calls[-1].kwargs['step'] == start + len(losses) * bs


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def log_second_phase(losses, batch_size, start_examples):
    examples_seen = start_examples
    for loss in losses:
        examples_seen += batch_size
        wandb.log({'loss': loss}, step=examples_seen)
    return examples_seen

wandb.log.reset_mock()
final = log_second_phase([0.5, 0.4, 0.3], batch_size=32, start_examples=640)
print('final examples_seen:', final)
print('first log step:', wandb.log.call_args_list[0].kwargs['step'])
```
</details>